# Primeros pasos con DataFrames de Spark

En el cuaderno anterior trabajamos con listas de valores. Aquí cada registro tiene varias columnas, como en una hoja de cálculo. Usaremos datos de accidentes de Barcelona para aprender a leer, filtrar y resumir una tabla.

Antes de empezar, completa la [guía del entorno de PySpark](../01_environment_pyspark.md) y ejecuta las celdas en orden.

## 1. Abrir una sesión de Spark

La sesión nos permite leer archivos y trabajar con sus datos. La interfaz web de Spark muestra el trabajo que ejecuta. Su dirección es distinta a la de Jupyter. Si trabajas con Jupyter o VS Code en la VM, o con Colab conectado a la VM, puedes abrir esa dirección en el navegador de la VM.

In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.master("local[2]").appName("DataFrames de Spark").getOrCreate()
print("Interfaz de Spark:", spark.sparkContext.uiWebUrl)

## 2. Leer el CSV y mirar la tabla

Usaremos el [CSV de accidentes de Barcelona](../data/2023_accidents_causa_conductor_gu_bcn_.csv), procedente de Open Data BCN. Si estás en Colab con el entorno de Google, sube el CSV mediante el panel **Archivos**. Si estás trabajando desde el repositorio en la VM, el cuaderno busca el archivo en la carpeta `data`.

`header=True` usa la primera fila como nombres de columnas. `inferSchema=True` permite que Spark deduzca cuáles son números y cuáles son texto.

In [ ]:
ruta = "../data/2023_accidents_causa_conductor_gu_bcn_.csv"

accidentes = spark.read.csv(ruta, header=True, inferSchema=True)
accidentes.select("Nom_districte", "Mes_any", "Hora_dia").show(5)
accidentes.printSchema()

## 3. Elegir columnas y filtrar filas

`select()` elige las columnas que queremos ver. `filter()` conserva las filas que cumplen una condición. Vamos a buscar registros de Sant Martí del primer semestre, es decir, de enero a junio.

In [ ]:
sant_marti = accidentes.filter(
    (F.col("Nom_districte") == "Sant Martí") & F.col("Mes_any").between(1, 6)
)
sant_marti.select("Nom_districte", "Nom_mes", "Hora_dia").show(5)

## 4. Resumir por distrito

`groupBy()` reúne las filas que tienen el mismo distrito y `count()` cuenta cuántos registros hay en cada grupo. Ordenamos el resultado para ver primero los distritos con más registros.

In [ ]:
registros_por_distrito = accidentes.groupBy("Nom_districte").count()
registros_por_distrito.orderBy(F.desc("count")).show()

## 5. Crear y unir dos tablas pequeñas

Imagina que tenemos una lista de pedidos y otra con el nombre de cada producto. Ambas contienen un código de producto. Usamos ese código para unirlas, como cuando buscamos el nombre de un artículo a partir de su código en una hoja de cálculo.

Después usamos `withColumn()` para calcular el importe de cada pedido.

In [ ]:
productos = spark.createDataFrame(
    [("P1", "Manzanas"), ("P2", "Pan")],
    ["codigo", "producto"],
)
pedidos = spark.createDataFrame(
    [("P1", 2, 1.5), ("P2", 3, 2.0), ("P1", 4, 1.5)],
    ["codigo", "unidades", "precio_unitario"],
)

detalle = pedidos.join(productos, on="codigo", how="inner").withColumn(
    "importe", F.col("unidades") * F.col("precio_unitario")
)
detalle.select("producto", "unidades", "importe").show()

## 6. Ahora prueba tú

Con el DataFrame `detalle`, agrupa los pedidos por producto y calcula el importe total de cada uno. Puedes combinar `groupBy()` con `sum("importe")`.

**Para comprobar tu resultado:** Manzanas suma 9 € y Pan suma 6 €.

In [ ]:
# Escribe aquí tu solución.

## 7. Cerrar Spark

Si abriste la interfaz web de Spark, puedes volver a ella antes de cerrar la sesión y mirar la pestaña **Jobs** para ver qué trabajos se ejecutaron.

In [ ]:
spark.stop()

## Navegación

[Volver al README](../README.md) · [Siguiente: Ejercicios de DataFrames](04_df_exercise.ipynb)